# 01 — Décorateurs

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre qu'un décorateur est une fonction qui enveloppe une autre fonction
- écrire un décorateur simple sans paramètre
- utiliser `functools.wraps` pour préserver le nom et la docstring
- écrire un décorateur paramétré (factory de décorateurs)
- utiliser des décorateurs de classe (`@classmethod`, `@staticmethod`, custom)
- empiler plusieurs décorateurs et comprendre l'ordre d'exécution
- connaître les décorateurs courants de la bibliothèque standard

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- toute l'Initiation 5j : types, fonctions typées, modules, fichiers
- le modèle objet (classes, héritage, `__str__`/`__repr__`)
- le typage moderne (`int | None`, `Protocol`, `Generic`)
- les dataclasses et Pydantic v2
- les closures (`04_Fonctions/03_Portee_et_closures`)

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- les context managers (notebook 02 de cette section)
- les design patterns (section 05)
- les metaclasses

## Plan

1. Fonctions comme objets de première classe
2. Premier décorateur : le concept
3. Syntaxe `@` : sucre syntaxique
4. `functools.wraps` : préserver les métadonnées
5. Décorateur avec arguments (factory)
6. Décorateurs de classe
7. Décorateur appliqué à une classe entière
8. Empiler des décorateurs : ordre d'exécution
9. Décorateurs de la stdlib : tour d'horizon
10. Pièges courants
11. Synthèse
12. Exercices

---

## 1. Fonctions comme objets de première classe

En Python, une fonction est un **objet** comme un autre. On peut la stocker dans une variable, la passer en argument, la retourner depuis une autre fonction.

In [ ]:
def saluer(nom: str) -> str:
    return f"Bonjour {nom}"


In [ ]:
type(saluer)


In [ ]:
ref = saluer
ref("Alice")


On peut passer une fonction en argument :

In [ ]:
def appliquer(func, valeur: str) -> str:
    return func(valeur)

appliquer(saluer, "Bob")


Et on peut retourner une fonction depuis une autre :

In [ ]:
def creer_salueur(formule: str):
    def saluer(nom: str) -> str:
        return f"{formule} {nom}"
    return saluer

bonjour = creer_salueur("Bonjour")
bonjour("Claire")


Ce dernier pattern — une fonction qui **retourne une fonction** — est la brique fondamentale du décorateur.

---

## 2. Premier décorateur : le concept

Un décorateur est une **fonction qui prend une fonction en argument** et **retourne une nouvelle fonction** qui enveloppe l'originale.

In [ ]:
def logger(func):
    def wrapper(*args, **kwargs):
        print(f"Appel de {func.__name__} avec args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} a retourné {result}")
        return result
    return wrapper


On l'applique manuellement :

In [ ]:
def addition(a: int, b: int) -> int:
    return a + b

addition_loggee = logger(addition)
addition_loggee(3, 5)


Le résultat est bien `8`, mais les deux `print` s'exécutent aussi. La fonction originale `addition` n'a **pas été modifiée**.

In [ ]:
addition(3, 5)  # pas de log ici


---

## 3. Syntaxe `@` : sucre syntaxique

Python offre la syntaxe `@decorateur` pour remplacer le pattern `func = decorateur(func)`. C'est exactement la même chose.

In [ ]:
@logger
def multiplication(a: int, b: int) -> int:
    return a * b


In [ ]:
multiplication(4, 7)


C'est strictement équivalent à :

```python
def multiplication(a, b):
    return a * b
multiplication = logger(multiplication)
```

### Problème : les métadonnées sont perdues

In [ ]:
multiplication.__name__


In [ ]:
multiplication.__doc__


Le `__name__` renvoie `wrapper` au lieu de `multiplication`. La docstring est celle du wrapper (aucune). C'est un problème pour le debug et la documentation.

---

## 4. `functools.wraps` : préserver les métadonnées

`functools.wraps` est un décorateur (!) qui copie le `__name__`, `__doc__`, `__module__` et `__qualname__` de la fonction originale sur le wrapper.

In [ ]:
from functools import wraps

def logger_v2(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Appel de {func.__name__}")
        return func(*args, **kwargs)
    return wrapper


In [ ]:
@logger_v2
def division(a: float, b: float) -> float:
    """Divise a par b."""
    return a / b


In [ ]:
division.__name__


In [ ]:
division.__doc__


**Règle absolue :** on met **toujours** `@wraps(func)` sur le wrapper. Sans exception.

---

## 5. Décorateur avec arguments (factory)

Souvent on veut **paramétrer** le décorateur. Il faut alors ajouter un niveau d'imbrication : une fonction qui retourne le décorateur.

In [ ]:
from functools import wraps

def repeter(n: int):
    """Décorateur qui appelle la fonction n fois."""
    def decorateur(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorateur


In [ ]:
@repeter(3)
def dire_bonjour(nom: str) -> str:
    print(f"Bonjour {nom}")
    return f"Bonjour {nom}"

dire_bonjour("Alice")


### Anatomie des 3 niveaux

| Niveau | Fonction | Rôle |
|--------|----------|------|
| 1 | `repeter(n)` | Reçoit le paramètre, retourne le décorateur |
| 2 | `decorateur(func)` | Reçoit la fonction à décorer |
| 3 | `wrapper(*args, **kwargs)` | Remplace l'appel original |

---

## 6. Décorateurs de classe

Une **méthode de classe** est un décorateur intégré à Python qui change le premier argument reçu.

### `@classmethod` et `@staticmethod` — rappel

| Décorateur | 1er arg | Usage typique |
|------------|---------|---------------|
| *(aucun)* | `self` (instance) | méthode normale |
| `@classmethod` | `cls` (la classe) | constructeur alternatif |
| `@staticmethod` | *(rien)* | utilitaire sans état |

In [ ]:
class Temperature:
    def __init__(self, celsius: float) -> None:
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, f: float) -> "Temperature":
        return cls((f - 32) * 5 / 9)

    @staticmethod
    def point_ebullition() -> float:
        return 100.0

    def __repr__(self) -> str:
        return f"Temperature({self.celsius:.1f}°C)"


In [ ]:
Temperature.from_fahrenheit(212)


In [ ]:
Temperature.point_ebullition()


---

## 7. Décorateur appliqué à une classe entière

Un décorateur peut aussi s'appliquer à une **classe**. Il reçoit la classe en argument et retourne une classe (souvent la même, modifiée).

In [ ]:
def auto_repr(cls):
    """Ajoute un __repr__ automatique à partir des annotations."""
    def __repr__(self) -> str:
        attrs = ", ".join(
            f"{k}={getattr(self, k)!r}"
            for k in cls.__annotations__
        )
        return f"{cls.__name__}({attrs})"
    cls.__repr__ = __repr__
    return cls


In [ ]:
@auto_repr
class Point:
    x: float
    y: float

    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

Point(1.5, 2.3)


C'est exactement ce que fait `@dataclass` (entre autres). Ce pattern est aussi utilisé par `@functools.total_ordering`.

---

## 8. Empiler des décorateurs : ordre d'exécution

On peut empiler plusieurs `@decorateur`. L'ordre d'application est **de bas en haut** (le plus proche de `def` est appliqué en premier).

In [ ]:
from functools import wraps

def gras(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper

def italique(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper


In [ ]:
@gras
@italique
def salut(nom: str) -> str:
    return f"Salut {nom}"

salut("Alice")


Explication : `salut` est d'abord emballé par `italique`, puis par `gras`. C'est équivalent à `gras(italique(salut))`.

In [ ]:
@italique
@gras
def salut2(nom: str) -> str:
    return f"Salut {nom}"

salut2("Bob")


L'ordre inverse produit `<i><b>...</b></i>` au lieu de `<b><i>...</i></b>`.

---

## 9. Décorateurs de la stdlib : tour d'horizon

Python fournit de nombreux décorateurs utiles dans la bibliothèque standard.

| Module | Décorateur | Usage |
|--------|-----------|-------|
| `functools` | `@wraps` | préserver les métadonnées |
| `functools` | `@lru_cache` | mise en cache (mémoïsation) |
| `functools` | `@singledispatch` | surcharge par type |
| `functools` | `@total_ordering` | comparaisons complètes |
| `dataclasses` | `@dataclass` | génération `__init__`, `__repr__`, etc. |
| `abc` | `@abstractmethod` | méthode abstraite |
| `contextlib` | `@contextmanager` | transformer un générateur en CM |
| `typing` | `@overload` | signatures multiples (typage) |

### `@lru_cache` : mémoïsation

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=128)
def fibonacci(n: int) -> int:
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

fibonacci(50)


In [ ]:
fibonacci.cache_info()


---

## 10. Pièges courants

Les erreurs les plus fréquentes avec les décorateurs.

### Piège 1 : oublier `@wraps`

In [ ]:
def mauvais_deco(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@mauvais_deco
def ma_fonction():
    """Ma docstring."""
    pass

print(ma_fonction.__name__)  # 'wrapper' au lieu de 'ma_fonction'
print(ma_fonction.__doc__)   # None au lieu de 'Ma docstring.'


### Piège 2 : oublier les parenthèses sur un décorateur paramétré

In [ ]:
# @repeter      # TypeError : repeter() missing argument
# @repeter(3)   # correct
# La différence : repeter reçoit la *fonction*, pas un entier.
print("Attention : @repeter et @repeter(3) sont très différents")


### Piège 3 : décorer avec un effet de bord au moment de l'import

In [ ]:
def enregistrer(func):
    print(f"Enregistrement de {func.__name__}")  # s'exécute à l'import !
    return func

@enregistrer
def traiter():
    pass
# Le print s'est exécuté maintenant, pas à l'appel de traiter()


Ce n'est pas toujours un piège — le pattern **Registry** (section 05) l'exploite délibérément.

---

## Synthèse

| Concept | Syntaxe | À retenir |
|---------|---------|----------|
| Décorateur simple | `def deco(func):` | Reçoit une fonction, retourne un wrapper |
| `@wraps` | `@wraps(func)` | **Toujours** sur le wrapper |
| Décorateur paramétré | `def deco(arg):` → `def decorateur(func):` → `def wrapper(...)` | 3 niveaux |
| Décorateur de classe | `def deco(cls):` | Modifie ou enveloppe une classe |
| Empilement | `@a` puis `@b` | Équivaut à `a(b(func))` |

### Règles à retenir

1. Un décorateur est une **fonction qui reçoit une fonction et retourne une fonction**.
2. Toujours utiliser `@functools.wraps(func)` sur le wrapper.
3. Décorateur paramétré = 3 niveaux d'imbrication.
4. L'empilement `@a` `@b` s'applique de bas en haut : `a(b(func))`.
5. Attention aux effets de bord au moment de la définition (import time).

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Timer *(facile)*

Écrivez un décorateur `timer` qui affiche le temps d'exécution d'une fonction.

```python
@timer
def slow() -> None:
    import time; time.sleep(0.1)

slow()  # slow a pris 0.10s
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Decorateurs", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} a pris {elapsed:.2f}s")
        return result
    return wrapper
```

</details>

### Exercice 2 — Compteur d'appels *(facile)*

Écrivez un décorateur `count_calls` qui compte le nombre d'appels à la fonction.
Le compteur est accessible via `func.call_count`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Decorateurs", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
from functools import wraps

def count_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.call_count += 1
        return func(*args, **kwargs)
    wrapper.call_count = 0
    return wrapper

@count_calls
def hello(name: str) -> str:
    return f"Hello {name}"

hello("A")
hello("B")
print(hello.call_count)  # 2
```

</details>

### Exercice 3 — Retry paramétré *(moyen)*

Écrivez un décorateur `retry(max_attempts: int, delay: float)` qui :

- rappelle la fonction jusqu'à `max_attempts` fois en cas d'exception
- attend `delay` secondes entre chaque tentative (`time.sleep`)
- lève l'exception si toutes les tentatives échouent

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Decorateurs", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import time
from functools import wraps

def retry(max_attempts: int = 3, delay: float = 0.5):
    def decorateur(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exc = e
                    if attempt < max_attempts:
                        time.sleep(delay)
            raise last_exc
        return wrapper
    return decorateur
```

</details>

### Exercice 4 — Validateur de types *(moyen)*

Écrivez un décorateur `validate_types` qui utilise les annotations de la fonction pour vérifier à l'exécution que chaque argument est du bon type. Levez `TypeError` si un argument ne correspond pas.

```python
@validate_types
def add(a: int, b: int) -> int:
    return a + b

add(1, 2)       # OK -> 3
add(1, "two")  # TypeError
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Decorateurs", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import inspect
from functools import wraps

def validate_types(func):
    sig = inspect.signature(func)
    hints = func.__annotations__

    @wraps(func)
    def wrapper(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)
        bound.apply_defaults()
        for name, value in bound.arguments.items():
            if name in hints and name != 'return':
                expected = hints[name]
                if not isinstance(value, expected):
                    raise TypeError(
                        f"{name} doit être {expected.__name__}, "
                        f"reçu {type(value).__name__}"
                    )
        return func(*args, **kwargs)
    return wrapper
```

</details>

---

## Ressources externes

### Documentation officielle
- [`functools.wraps`](https://docs.python.org/3/library/functools.html#functools.wraps)
- [`functools.lru_cache`](https://docs.python.org/3/library/functools.html#functools.lru_cache)
- [Decorator glossary](https://docs.python.org/3/glossary.html#term-decorator)

### PEPs de référence
- [PEP 318 — Decorators for Functions and Methods](https://peps.python.org/pep-0318/)
- [PEP 3129 — Class Decorators](https://peps.python.org/pep-3129/)

### Lectures complémentaires
- Fluent Python, ch. 9 « Decorators and Closures »
- Python Cookbook, ch. 9 « Metaprogramming »